# Candidate detection: PlanetScope small vessels (Nuku'alofa)

Complete pipeline, top to bottom:
1. Group scenes into acquisitions (split passes mosaicked)
2. Cloud mask (UDM2 + buffer) and per-acquisition usability
3. Detection water: WorldCover water, widened, restricted to an **analysis zone** you draw (harbour + approaches, no reef flats/surf)
4. Background: low percentile (p25 by default) of the background acquisitions, so boats docked on most dates still stand out
5. Score: NIR anomaly relative to the background, scaled by per-pixel noise **and** per-scene noise
6. Blobs → candidate points → exported to the shared asset + Drive GeoJSON

Exports are deliberately loose (`EXPORT_THRESH`); pick the final cut-off on `score_max` in step 9 without re-exporting.

## 1. Settings

In [2]:
GEE_PROJECT = "groupprojectgeog761"
COLLECTION = "planet_nukualofa_wharf"

# --- cloud screening ---
BUFFER_M = 30            # grow cloud/shadow mask (cloud edges cause false blobs)
KEEP_LIGHT_HAZE = True   # heavy haze is always masked
MIN_USABLE = 0.30        # share of clear AOI water below which an acquisition is dropped
GOOD_USABLE = 0.80
MIN_CLEAR_OBS = 5        # pixels with fewer clear background dates are excluded from detection

# Maxar-date scenes: kept OUT of the background, still run through detection
EXCLUDE_FROM_MEDIAN = [
    "20220114_210441_72_2223",
    "20220114_210439_51_2223",
    "20220118_213938_1026",
]
TEST_ACQ = []            # held-out test acquisition IDs (e.g. "20240701_24cf"): tagged role="test"

# --- detection area ---
ZONE_FILE = "analysis_zone.geojson"   # zone polygons from QGIS (field "zone"), any CRS; None = whole AOI
EXTRA_WATER_FILE = "analysis_zone.geojson"               # optional polygons forced to count as water (harbour basins)
WATER_BUFFER_M = 20                   # widen WorldCover water so boats at the wharf aren't cut off

# --- detection ---
BG_PERCENTILE = 25       # 50 = median; 25 lets boats docked on up to ~75% of dates stand out
SCORE_BANDS = "NIR"      # "NIR" (recommended) or "NIR+VIS"
EXPORT_THRESH = 6.0      # loose export threshold; final cut chosen in step 9
MAX_BLOB_PX = 150        # larger blobs dropped (glint, cloud remnants)

CAND_NAME = f"candidates_{COLLECTION}"
CAND_ASSET = f"projects/{GEE_PROJECT}/assets/{CAND_NAME}"

## 2. Setup
Colab already has geemap. If you ever install it with pip, restart the session afterwards (Runtime → Restart session).

In [3]:
import re, json, time
from pathlib import Path
from collections import defaultdict
from datetime import datetime
import ee, geemap
import pandas as pd
import geopandas as gpd

ee.Authenticate()
ee.Initialize(project=GEE_PROJECT)

col = ee.ImageCollection(f"projects/{GEE_PROJECT}/assets/{COLLECTION}")
print("Scenes:", col.size().getInfo())
print("Bands:", col.first().bandNames().getInfo())

*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_9oS0DRcPvElRMNw?source=python


Scenes: 29
Bands: ['B1', 'B2', 'B3', 'B4', 'Q1', 'Q2', 'Q3', 'Q4', 'Q5', 'Q6', 'Q7', 'Q8']


## 3. Group scenes into acquisitions
Same date + same satellite = one acquisition. All bands (incl. UDM2) are cut to each scene's footprint before mosaicking.

In [4]:
SCENE_RE = re.compile(r"^(\d{8})_(\d{6})(?:_\d{1,2})?_([0-9a-f]{4})")

asset_ids = col.aggregate_array("system:index").getInfo()
groups, scene_of = defaultdict(list), {}
for a in asset_ids:
    m = SCENE_RE.match(a)
    if not m:
        print("Unrecognised asset name, skipped:", a)
        continue
    scene_of[a] = m.group(0)
    groups[f"{m.group(1)}_{m.group(3)}"].append(a)

for k, v in groups.items():
    if len(v) > 1:
        print(f"Mosaicking {k}: {[scene_of[a] for a in v]}")
print(f"{len(asset_ids)} scenes -> {len(groups)} acquisitions")

def build_acq(acq_id, assets):
    first = col.filter(ee.Filter.eq("system:index", assets[0])).first()
    proj = first.select("B1").projection()
    parts = col.filter(ee.Filter.inList("system:index", assets)).map(
        lambda i: i.updateMask(i.select("B1").gt(0)))
    img = parts.mosaic().setDefaultProjection(proj)
    m = SCENE_RE.match(assets[0])
    date = datetime.strptime(m.group(1) + m.group(2), "%Y%m%d%H%M%S").strftime("%Y-%m-%d %H:%M")
    return img.set({"acq_id": acq_id,
                    "scene_ids": ",".join(sorted(scene_of[a] for a in assets)),
                    "date": date})

acq = ee.ImageCollection([build_acq(k, v) for k, v in sorted(groups.items())])

exclude_acq = sorted({k for k, v in groups.items() if any(scene_of[a] in EXCLUDE_FROM_MEDIAN for a in v)})
missing = [s for s in EXCLUDE_FROM_MEDIAN if s not in scene_of.values()]
print("Excluded from background:", exclude_acq)
if missing:
    print("WARNING - not in the collection:", missing)

Mosaicking 20190623_1054: ['20190623_213507_1054', '20190623_213508_1054']
Mosaicking 20220114_2223: ['20220114_210439_51_2223', '20220114_210441_72_2223']
29 scenes -> 27 acquisitions
Excluded from background: ['20220114_2223', '20220118_1026']


## 4. AOI, water and cloud mask

In [6]:
aoi = col.map(lambda i: ee.Feature(i.geometry())).union(10).geometry()
water = (ee.ImageCollection("ESA/WorldCover/v200").first().eq(80)
         .unmask(0).clip(aoi).rename("water"))

def mask_scene(img):
    data = img.select("B1").gt(0).unmask(0)
    not_clear = img.select("Q1").eq(0)
    if KEEP_LIGHT_HAZE:
        not_clear = not_clear.And(img.select("Q4").eq(0))
    bad = (not_clear.Or(img.select("Q3").eq(1))
                    .Or(img.select("Q5").eq(1))
                    .Or(img.select("Q6").eq(1))).unmask(0).And(data)
    bad_buf = bad.focalMax(radius=BUFFER_M, units="meters", kernelType="circle")
    valid = data.And(bad_buf.Not())
    return img.addBands(data.rename("data")).addBands(valid.rename("valid"))

def masked(img):
    return img.select(["B1", "B2", "B3", "B4"]).updateMask(img.select("valid"))

col_m = acq.map(mask_scene)

## 5. Usability per acquisition

In [8]:
import geopandas as gpd
from pathlib import Path

def load_geom(path):
    if path is None:
        return None
    if not Path(path).exists():
        from google.colab import files
        print(f"Upload {path}:")
        files.upload()
    return geemap.gdf_to_ee(gpd.read_file(path).to_crs(4326)).geometry()

zone = load_geom(ZONE_FILE)
stat_water = water.clip(zone).unmask(0).rename("water") if zone is not None else water

def acq_stats(img):
    proj = img.select("B1").projection()
    s = ee.Image.cat(
        stat_water,
        img.select("data").And(stat_water).rename("data_water"),
        img.select("valid").And(stat_water).rename("valid_water"),
    ).reduceRegion(ee.Reducer.sum(), aoi, crs=proj, scale=3, maxPixels=1e9)
    return ee.Feature(None, s).copyProperties(img, ["acq_id", "scene_ids", "date"])

fc = ee.FeatureCollection(col_m.map(acq_stats))
try:
    df = ee.data.computeFeatures({"expression": fc, "fileFormat": "PANDAS_DATAFRAME"})
except Exception:
    df = geemap.ee_to_df(fc)

df["footprint_frac"] = df.data_water / df.water
df["usable_frac"] = df.valid_water / df.water
df["category"] = pd.cut(df.usable_frac, [-1, MIN_USABLE, GOOD_USABLE, 2],
                        labels=["drop", "partial", "clear"], right=False).astype(str)
df["role"] = df.acq_id.map(lambda a: "maxar_eval" if a in exclude_acq
                           else ("test" if a in TEST_ACQ else "background"))
out = (df[["acq_id", "scene_ids", "date", "role", "footprint_frac", "usable_frac", "category"]]
       .sort_values("date").round(3).reset_index(drop=True))
out.to_csv(f"scene_cloud_stats_{COLLECTION}.csv", index=False)

usable = out.loc[out.category != "drop", "acq_id"].tolist()
bg_ids = [a for a in usable if a not in exclude_acq and a not in TEST_ACQ]
target_ids = sorted(set(usable) | set(exclude_acq))       # Maxar dates always detected on
col_bg = col_m.filter(ee.Filter.inList("acq_id", bg_ids))
col_targets = col_m.filter(ee.Filter.inList("acq_id", target_ids))

print(pd.crosstab(out.role, out.category))
print(f"\nBackground: {len(bg_ids)} | detection targets: {len(target_ids)}")

Upload analysis_zone.geojson:


Saving analysis_zone.geojson to analysis_zone.geojson
category    clear  partial
role                      
background     25        0
maxar_eval      1        1

Background: 25 | detection targets: 27


## 6. Detection water
WorldCover water, widened by `WATER_BUFFER_M`, plus optional basin polygons, restricted to the analysis zone and to pixels with at least `MIN_CLEAR_OBS` clear background dates.

**Analysis zone:** in QGIS draw one polygon over the harbour, wharf basins and boat approaches; leave out reef flats, the surf line and open sea far from the harbour. Any CRS works (it is reprojected here). You will be asked to upload it if it isn't already in the Colab session.

In [7]:
def load_geom(path):
    if path is None:
        return None
    if not Path(path).exists():
        from google.colab import files
        print(f"Upload {path}:")
        files.upload()
    gdf = gpd.read_file(path).to_crs(4326)      # any CRS (e.g. EPSG:32701) or GeoPackage works
    return geemap.gdf_to_ee(gdf).geometry()

zone = load_geom(ZONE_FILE)
extra = load_geom(EXTRA_WATER_FILE)

n_all = col_bg.map(lambda i: i.select("valid")).sum().unmask(0)
det_water = water.focalMax(radius=WATER_BUFFER_M, kernelType="circle", units="meters")
if extra is not None:
    det_water = det_water.Or(ee.Image(1).clip(extra).unmask(0))
det_water = det_water.And(n_all.gte(MIN_CLEAR_OBS))
if zone is not None:
    det_water = det_water.clip(zone)
det_water = det_water.selfMask().rename("det_water")

km2 = det_water.multiply(ee.Image.pixelArea()).reduceRegion(
    ee.Reducer.sum(), aoi, 10, maxPixels=1e9).getNumber("det_water").divide(1e6).getInfo()
print(f"Detection water: {km2:.2f} km²")

Detection water: 41.29 km²


## 7. Background and noise

In [9]:
def _num(d, k):
    return ee.Number(ee.Algorithms.If(d.get(k), d.get(k), 0))

def normalise(img):
    """VIS = mean(B1..B3), NIR = B4, each minus the scene's own median over detection water."""
    x = (img.select(["B1", "B2", "B3"]).reduce(ee.Reducer.mean()).rename("VIS")
         .addBands(img.select("B4").rename("NIR"))
         .updateMask(det_water))
    med = x.reduceRegion(ee.Reducer.median(), aoi, 9, bestEffort=True, maxPixels=1e8)
    off = ee.Image.constant([_num(med, "VIS"), _num(med, "NIR")]).rename(["VIS", "NIR"])
    return ee.Image(x.subtract(off).toFloat().copyProperties(img, ["acq_id", "scene_ids", "date"]))

bg_n = col_bg.map(masked).map(normalise)
P = BG_PERCENTILE
pct = bg_n.reduce(ee.Reducer.percentile(sorted({25, 50, 75, P})))
bg_ref = pct.select([f"VIS_p{P}", f"NIR_p{P}"], ["VIS", "NIR"])
sigma = (pct.select(["VIS_p75", "NIR_p75"]).subtract(pct.select(["VIS_p25", "NIR_p25"]))
         .divide(1.349).rename(["VIS", "NIR"]))
sigma = sigma.focalMedian(radius=10, kernelType="circle", units="meters")   # smooth the per-pixel noise estimate
floor = sigma.updateMask(det_water).reduceRegion(ee.Reducer.median(), aoi, 9, bestEffort=True, maxPixels=1e8)
sigma = sigma.max(ee.Image.constant([_num(floor, "VIS"), _num(floor, "NIR")]).rename(["VIS", "NIR"]))
cap = ee.Image.constant([_num(floor, "VIS").multiply(3),
                         _num(floor, "NIR").multiply(3)]).rename(["VIS", "NIR"])
sigma = sigma.min(cap)    # restless areas get at most 3x the typical noise
floor = floor.getInfo()
print("Typical water noise (normalised units):", floor)

Typical water noise (normalised units): {'NIR': 20.21336402037735, 'VIS': 19.818043448768474}


## Export background model

In [10]:
BG_ASSET = f"projects/{GEE_PROJECT}/assets/bgmodel_{COLLECTION}"
bgmodel = ee.Image.cat(
    det_water.unmask(0).rename("det_water"),
    bg_ref.rename(["bg_VIS", "bg_NIR"]),
    sigma.rename(["sig_VIS", "sig_NIR"]),
).toFloat()
try:
    ee.data.deleteAsset(BG_ASSET)
except ee.EEException:
    pass
ee.batch.Export.image.toAsset(
    image=bgmodel, description="bgmodel_to_asset", assetId=BG_ASSET,
    region=aoi, crs="EPSG:32701", scale=3, maxPixels=1e10,
    pyramidingPolicy={".default": "mean", "det_water": "max"}).start()
print("Started - wait for COMPLETED before step 9")

Started - wait for COMPLETED before step 9


## 8. Check before exporting
Cyan should cover the harbour basins and approaches, not reef flats. Switch on clear observations and noise to see weak areas.

In [11]:
mw = geemap.Map(basemap="Esri.WorldImagery")
mw.addLayer(det_water, {"palette": ["00ffff"], "opacity": 0.35}, "detection water")
mw.addLayer(n_all.updateMask(det_water), {"min": 0, "max": max(len(bg_ids), 1),
            "palette": ["ff0000", "ffff00", "00ff00"]}, "clear background dates", False)
mw.addLayer(sigma.select("NIR").updateMask(det_water),
            {"min": 0, "max": 3 * (floor.get("NIR") or 1), "palette": ["000000", "ffffff"]}, "NIR noise", False)
if zone is not None:
    mw.addLayer(ee.FeatureCollection([ee.Feature(zone)]).style(color="ffff00", fillColor="00000000"), {}, "analysis zone")
mw.centerObject(aoi, 14)
mw

## Use the stored model

In [26]:
def _num(d, k):
    return ee.Number(ee.Algorithms.If(d.get(k), d.get(k), 0))

def normalise(img):
    x = (img.select(["B1", "B2", "B3"]).reduce(ee.Reducer.mean()).rename("VIS")
         .addBands(img.select("B4").rename("NIR"))
         .updateMask(det_water))
    med = x.reduceRegion(ee.Reducer.median(), aoi, 9, bestEffort=True, maxPixels=1e8, tileScale=4)
    off = ee.Image.constant([_num(med, "VIS"), _num(med, "NIR")]).rename(["VIS", "NIR"])
    return ee.Image(x.subtract(off).toFloat().copyProperties(img, ["acq_id", "scene_ids", "date"]))

BG_ASSET = f"projects/{GEE_PROJECT}/assets/bgmodel_{COLLECTION}"
bm = ee.Image(BG_ASSET)
det_water = bm.select("det_water").selfMask().rename("det_water")
bg_ref = bm.select(["bg_VIS", "bg_NIR"], ["VIS", "NIR"])
sigma = bm.select(["sig_VIS", "sig_NIR"], ["VIS", "NIR"])
print("Using stored background model:", bm.bandNames().getInfo())

Using stored background model: ['det_water', 'bg_VIS', 'bg_NIR', 'sig_VIS', 'sig_NIR']


## 9. Detect and export

In [19]:
def score_image(img):
    z = img.subtract(bg_ref).divide(sigma)
    # per-scene centring and scaling: rough/glinty scenes get a stricter effective threshold
    zs = z.updateMask(det_water).reduceRegion(ee.Reducer.percentile([25, 50, 75]), aoi, 9,
                                              bestEffort=True, maxPixels=1e8, tileScale=4)
    def p(b, q): return _num(zs, f"{b}_p{q}")
    centre = ee.Image.constant([p("VIS", 50), p("NIR", 50)]).rename(["VIS", "NIR"])
    spread = ee.Image.constant([p("VIS", 75).subtract(p("VIS", 25)).divide(1.349).max(1),
                                p("NIR", 75).subtract(p("NIR", 25)).divide(1.349).max(1)]).rename(["VIS", "NIR"])
    z = z.subtract(centre).divide(spread)
    score = z.select("NIR") if SCORE_BANDS == "NIR" else z.select("NIR").max(z.select("VIS"))
    return score.rename("score").updateMask(det_water)

def detect(img):
    score = score_image(img)
    hit = score.gt(EXPORT_THRESH).selfMask()
    blobs = hit.int().connectedComponents(ee.Kernel.square(1), MAX_BLOB_PX).select("labels")
    vec = blobs.addBands(score).reduceToVectors(
        geometry=aoi, crs=img.select("NIR").projection(), scale=3,
        geometryType="centroid", eightConnected=True, labelProperty="blob",
        reducer=ee.Reducer.max().combine(ee.Reducer.count(), sharedInputs=True),
        maxPixels=1e9, tileScale=4)
    acq_id = img.get("acq_id")
    role = ee.Algorithms.If(ee.List(exclude_acq).contains(acq_id), "maxar_eval",
           ee.Algorithms.If(ee.List(TEST_ACQ).contains(acq_id), "test", "train"))
    return vec.map(lambda f: f.set({
        "acq_id": acq_id, "scene_ids": img.get("scene_ids"), "date": img.get("date"),
        "role": role, "score_max": f.get("max"), "area_px": f.get("count")}))

cands_fc = ee.FeatureCollection(col_targets.map(masked).map(normalise).map(detect)).flatten()

try:
    ee.data.deleteAsset(CAND_ASSET)
    print("Deleted old", CAND_ASSET)
except ee.EEException:
    pass

ee.batch.Export.table.toAsset(collection=cands_fc, description="candidates_to_asset", assetId=CAND_ASSET).start()
ee.batch.Export.table.toDrive(collection=cands_fc, description="candidates_to_drive",
                              folder="planet_exports", fileNamePrefix=CAND_NAME, fileFormat="GeoJSON").start()
print("Exports started.")

Exports started.


Wait for the exports (safe to close the laptop; re-run steps 1–8, skip 9, then continue here).

In [27]:
while True:
    st = {}
    for t in ee.batch.Task.list()[:10]:
        s = t.status()
        if s["description"].startswith("candidates_") and s["description"] not in st:
            st[s["description"]] = s
    print(time.strftime("%H:%M:%S"), {k: v["state"] for k, v in st.items()})
    if st and all(v["state"] in ("COMPLETED", "FAILED", "CANCELLED") for v in st.values()):
        for k, v in st.items():
            if v["state"] == "FAILED":
                print(k, "->", v.get("error_message"))
        break
    time.sleep(30)

03:29:23 {'candidates_to_drive': 'COMPLETED', 'candidates_to_asset': 'COMPLETED'}


## 10. Choose the cut-off
Pick the threshold where totals are manageable **and** the boats noted in the team sheet still have a candidate (check in QGIS with the filter `"score_max" >= T`).

In [28]:
cands = ee.FeatureCollection(CAND_ASSET)
rows = []
for t in [6, 8, 10, 12, 15, 20]:
    f = cands.filter(ee.Filter.gte("score_max", t))
    rows.append({"score >=": t, "candidates": f.size().getInfo(),
                 "acquisitions with any": len(f.aggregate_histogram("acq_id").getInfo())})
print(pd.DataFrame(rows).to_string(index=False))

 score >=  candidates  acquisitions with any
        6       22193                     27
        8        5278                     27
       10        1731                     27
       12         815                     26
       15         477                     25
       20         298                     24


Per zone: harbour and open water usually need different cut-offs (open water is choppier).

In [29]:
zones_fc = geemap.gdf_to_ee(gpd.read_file(ZONE_FILE).to_crs(4326)) if ZONE_FILE else None

def add_zone(f):
    z = zones_fc.filterBounds(f.geometry()).first()
    return f.set("zone", ee.Algorithms.If(z, ee.Feature(z).get("zone"), "none"))

if zones_fc is not None:
    cz = cands.map(add_zone)
    rows = []
    for t in [6, 8, 10, 12, 15, 20]:
        h = cz.filter(ee.Filter.gte("score_max", t)).aggregate_histogram("zone").getInfo()
        rows.append({"score >=": t, **h})
    print(pd.DataFrame(rows).fillna(0).astype(int).to_string(index=False))

 score >=  harbour  none  open  shore
        6      952   240 20877    124
        8      505    73  4640     60
       10      375    37  1276     43
       12      300    29   454     32
       15      233    22   198     24
       20      157    18   106     17


In [30]:
CUT = 12   # set after looking at the table and QGIS
per_acq = pd.Series(cands.filter(ee.Filter.gte("score_max", CUT)).aggregate_histogram("acq_id").getInfo())
print(f"Candidates per acquisition at score >= {CUT}:")
print(per_acq.reindex(target_ids).fillna(0).astype(int).sort_values(ascending=False).to_string())

Candidates per acquisition at score >= 12:
20190808_1014    137
20240812_24d0     68
20190623_1054     63
20210421_105a     62
20240602_2424     45
20240701_24cf     40
20240712_24f9     34
20190821_1069     31
20250110_24dd     31
20240520_24c9     30
20200510_106c     29
20260329_252e     27
20240518_24f1     26
20240722_24e8     26
20220812_249c     24
20260607_2575     20
20211129_2430     20
20210722_1065     19
20251013_24db     17
20221216_2495     15
20211222_2403     13
20210927_2464     12
20220114_2223      8
20250910_252d      7
20250828_2508      7
20200108_105c      4
20220118_1026      0


## 11. Look at one acquisition: image, score and candidates

In [32]:
A = "20240602_2424"
img = col_targets.filter(ee.Filter.eq("acq_id", A)).first()
sc = score_image(normalise(masked(img)))
mc = geemap.Map(basemap="Esri.WorldImagery")
mc.addLayer(img, {"bands": ["B3", "B2", "B1"], "min": 0, "max": 3000}, "RGB")
mc.addLayer(img, {"bands": ["B4"], "min": 0, "max": 3000}, "NIR", False)
mc.addLayer(sc, {"min": 0, "max": 15, "palette": ["000000", "ffffff", "ff0000"]}, "score", False)
mc.addLayer(cands.filter(ee.Filter.And(ee.Filter.eq("acq_id", A), ee.Filter.gte("score_max", CUT))),
            {"color": "ff00ff"}, f"candidates >= {CUT}")
mc.centerObject(aoi, 15)
mc

## 12. Optional: export the acquisitions as GeoTIFFs for labelling in QGIS
One file per acquisition (split passes merged), B1–B4, uint16, EPSG:32701, ~30–40 MB each, to Drive `planet_scenes`. Move them to the Shared Drive afterwards. Only run once.

In [10]:
EXPORT_SCENES = True
if EXPORT_SCENES:
    for a in target_ids:
        im = col_m.filter(ee.Filter.eq("acq_id", a)).first()
        ee.batch.Export.image.toDrive(
            image=im.select(["B1", "B2", "B3", "B4"]).toUint16(),
            description=f"ps_{a}", folder="planet_scenes", fileNamePrefix=f"ps_{a}",
            region=aoi, crs="EPSG:32701", scale=3, maxPixels=1e9, formatOptions={"cloudOptimized": True}).start()
    print(f"Started {len(target_ids)} scene exports")

Started 27 scene exports


In [15]:
from collections import Counter
print(Counter(t.status()["state"] for t in ee.batch.Task.list()
              if t.status()["description"].startswith("ps_")))

Counter({'COMPLETED': 27})


In [16]:
import geopandas as gpd, numpy as np
from google.colab import drive, files
drive.mount("/content/drive")

CAND = "/content/drive/MyDrive/planet_exports/candidates_planet_nukualofa_wharf.geojson"
OUT = "/content/drive/MyDrive/tonga-vessels-data/labels/ps_labels.gpkg"
CUTS = {"harbour": 8, "open": 12}     # your cut-offs from step 10
TEST_ACQ = []                         # 3-5 acquisitions held out for exhaustive labelling

if not Path("analysis_zone.geojson").exists():
    files.upload()
C = gpd.read_file(CAND).to_crs(32701)
Z = gpd.read_file("analysis_zone.geojson").to_crs(32701)[["zone", "geometry"]]
C = gpd.sjoin(C, Z, how="left", predicate="within").drop(columns="index_right")
L = C[C.apply(lambda r: r.score_max >= CUTS.get(r.zone, 1e9), axis=1) & ~C.acq_id.isin(TEST_ACQ)].copy()

for f in ["class", "confidence", "notes"]:
    L[f] = ""
L["n_hulls"], L["length_m"] = np.nan, np.nan
L["labeller"], L["source"] = "JE", "candidate"
L["set"] = np.where(L.role == "maxar_eval", "maxar_eval", "train")
L.to_file(OUT, layer="ps_labels", driver="GPKG")
print(len(L), "candidates"); print(L.groupby("acq_id").size().to_string())

Mounted at /content/drive
960 candidates
acq_id
20190623_1054     84
20190808_1014    156
20190821_1069     42
20200108_105c      4
20200510_106c     48
20210421_105a     61
20210722_1065     19
20210927_2464     11
20211129_2430     33
20211222_2403     26
20220114_2223     10
20220812_249c     31
20221216_2495     14
20240518_24f1     26
20240520_24c9     40
20240602_2424     46
20240701_24cf     47
20240712_24f9     36
20240722_24e8     31
20240812_24d0     79
20250110_24dd     36
20250828_2508      8
20250910_252d      8
20251013_24db     18
20260329_252e     27
20260607_2575     19
